#### <span style="background-color:pink;">SCD Type 2 Production Template</span> ####
Below is a **clean, production-ready, Databricks/Fabric** notebook template that performs **Slowly Changing Dimension Type 2 (SCD2)** handling using **Delta Lake MERGE**.
This is the **industry-standard** implementation used in enterprise lakehouse architectures.

It supports:

* **Insert new dimension rows**
* **Expire old rows** when a change is detected
* **Insert new SCD2 version** with updated fields
* **Natural key–based matching**
* **Versioning and effective dates**

You can use this notebook as a **production template** immediately.

#### **SCD Type 2 — Delta Lake Notebook Template** ####

##### 1. **Parameters** #####



##### <mark>If you want to get current Lakehouse info (friendly name)</mark>

In [25]:
import os
from notebookutils import mssparkutils

print(os.getcwd())

row = spark.sql("SELECT current_catalog(), current_database()").collect()[0]
print(row)

catalog = row[0]
lakehouse = row[1]

current_lakehouse_id = row[0]
print("ID : ", current_lakehouse_id)

print("Catalog:", catalog)
print("Lakehouse Name:", lakehouse)

#print("Lakehouse ID:", lakehouse_id)
#print("Lakehouse Name:", lh["name"])

# from config setting
#print(spark.conf.get("spark.sql.catalogImplementation"))
#print(spark.conf.get("spark.sql.defaultDatabase"))                      #not called in fabric, fails





StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 27, Finished, Available, Finished, False)

/mnt/var/hadoop/tmp/nm-local-dir/usercache/trusted-service-user/appcache/application_1770245392320_0001/container_1770245392320_0001_01_000001
Row(current_catalog()='spark_catalog', current_database()='laserengravelakehouse')
ID :  spark_catalog
Catalog: spark_catalog
Lakehouse Name: laserengravelakehouse


In [26]:
row = spark.sql("SELECT current_database()").collect()[0]
print("Spark Lakehouse ID:", row[0])

from notebookutils import mssparkutils

lakehouse_id = spark.sql(
    "SELECT current_database()"
).collect()[0][0]

lh = mssparkutils.lakehouse.get(lakehouse_id)

#print("Lakehouse name:", lh["name"])
print("Lakehouse id:", lh["id"])
print("Workspace id:", lh["workspaceId"])



StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 28, Finished, Available, Finished, False)

Spark Lakehouse ID: laserengravelakehouse
Lakehouse id: 8af43c6b-4303-4b0b-8459-7621117771f2
Workspace id: 4ecbaf18-d748-4bd3-b6cd-5743f135f2bf


##### If you want to get current Lakehouse info (friendly name)

#### Source table (staging area / raw feed) and Target Location

In [27]:
source_path = "Files/staging/customers"

# Target SCD2 dimension table
target_path = "Files/dim/customer_dim"
target_table = "dim_customer"

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 29, Finished, Available, Finished, False)

##### <mark>Business Key (could be a compound key - multiple columns for row uniqueness)</mark>

In [28]:
# Natural key for SCD2
business_key = "customer_id"

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 30, Finished, Available, Finished, False)

##### <span style="background-color:pink;"> Type 2 SCD Change Columns to track - So any change in any of these columns would flag a new 'insert' for the customer 
##### <mark>and "expire" their previous row (populate effective_end date</mark>

In [29]:
# Columns used to detect changes (non-key columns)
scd_columns = ["customer_name", "customer_status", "address"]

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 31, Finished, Available, Finished, False)

In [30]:
# Metadata columns for SCD2
effective_start = "effective_start_date"
effective_end = "effective_end_date"
is_current = "is_current"

now = "current_timestamp()"

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 32, Finished, Available, Finished, False)


#### 2. Load Incoming Staging Data - load to memory in temporary table ####

In [31]:
df_source = spark.read.format("delta").load(source_path).orderBy("customer_id")
df_source.createOrReplaceTempView("src")

print("Source count:", df_source.count())

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 33, Finished, Available, Finished, False)

Source count: 200


##### <mark>Let's see what we've staged in the temp table now </mark>

In [32]:
df_temp = spark.sql("SELECT * FROM src LIMIT 1000").orderBy("customer_id")
display(df_temp)

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35c3ff7b-984e-469a-b5c6-fdd2318d9a40)

##### <mark>3. Debug - table locations ####

In [33]:
print("target table:", target_table)
print("target table path:", target_path)

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 35, Finished, Available, Finished, False)

target table: dim_customer
target table path: Files/dim/customer_dim


#### <span style="background-color:pink">Create target delta table if does not exist</span>

In [34]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_table} (
    customer_id STRING,
    customer_name STRING,
    customer_status STRING,
    address STRING,
    {effective_start} TIMESTAMP,
    {effective_end} TIMESTAMP,
    {is_current} BOOLEAN
)
USING DELTA
""")

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 36, Finished, Available, Finished, False)

DataFrame[]

#### 4. Define MERGE Logic (SCD2 Core Logic) ####
<mark>This is the **production pattern**.</mark>

The 
" <mark>OR</mark> ".join(...):  "OR" is specified as the part that joins the iterated values returned from the list comprehension (expression resolved from the list) 

Concatenates them into one single string, inserting " OR " between each element

**"tgt.customer_name <> src.customer_name <mark>OR</mark> tgt.address <> src.address <mark>OR</mark> tgt.status <> src.status"**



In [35]:
spark.sql(f"""
MERGE INTO {target_table} AS tgt
USING (
    SELECT
        *,
        current_timestamp() AS update_ts
    FROM src
) AS src
ON tgt.{business_key} = src.{business_key} AND tgt.{is_current} = TRUE

WHEN MATCHED AND (
    {" OR ".join([f"tgt.{c} <> src.{c}" for c in scd_columns])}     
)
THEN UPDATE SET
    tgt.{effective_end} = src.update_ts,
    tgt.{is_current} = FALSE

WHEN NOT MATCHED THEN INSERT (
    {business_key},
    {", ".join(scd_columns)},
    {effective_start},
    {effective_end},
    {is_current}
)
VALUES (
    src.{business_key},
    {", ".join([f"src.{c}" for c in scd_columns])},
    src.update_ts,
    NULL,
    TRUE
)
""")


StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 37, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [36]:
spark.sql("""
SELECT *
FROM dim_customer
ORDER BY customer_id
""").show()

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 38, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+--------------------+--------------------+----------+
|customer_id|customer_name|customer_status|   address|effective_start_date|  effective_end_date|is_current|
+-----------+-------------+---------------+----------+--------------------+--------------------+----------+
|      C0001|Customer_0001|         Active| 1 Main St|2025-12-31 00:55:...|                NULL|      true|
|      C0002|Customer_0002|       Inactive| 2 Main St|2025-12-31 00:55:...|                NULL|      true|
|      C0003|Customer_0003|         Active| 3 Main St|2025-12-31 00:55:...|                NULL|      true|
|      C0004|Customer_0004|         Active| 4 Main St|2025-12-31 00:55:...|                NULL|      true|
|      C0005|Customer_0005|         Active| 5 Main St|2025-12-31 00:55:...|2026-02-04 23:16:...|     false|
|      C0006|Customer_0006|         Active| 6 Main St|2025-12-31 00:55:...|2026-02-04 23:16:...|     false|
|      C0007|Customer_0007| 

#### 5. **SCD2 Logic Explained Quickly** ####

##### ✔ When values change: #####

1. The current record is **closed**

<mark>   * `effective_end_date = current_timestamp()`</mark>
   * `is_current = false`

2. A new record is **inserted**

<mark>   * `effective_start_date = current_timestamp()`
   * `is_current = true`</mark>
##### ✔ When values do NOT change: #####

* No modifications are made.

##### ✔ When a business key does not exist: #####

* A new SCD2 record is inserted.

### 6. **Check Final Results**

In [37]:
df_dim = spark.read.format("delta").load(target_path)
display(df_dim.orderBy("customer_id", effective_start))

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2f161e7-3f02-4503-9ad0-01651ca73539)

### **Check statuses** ### 

In [38]:
spark.sql("""
SELECT *
FROM dim_customer
WHERE is_current = TRUE
ORDER BY customer_id, effective_start_date
""").show()

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 40, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+--------------------+------------------+----------+
|customer_id|customer_name|customer_status|   address|effective_start_date|effective_end_date|is_current|
+-----------+-------------+---------------+----------+--------------------+------------------+----------+
|      C0001|Customer_0001|         Active| 1 Main St|2025-12-31 00:55:...|              NULL|      true|
|      C0002|Customer_0002|       Inactive| 2 Main St|2025-12-31 00:55:...|              NULL|      true|
|      C0003|Customer_0003|         Active| 3 Main St|2025-12-31 00:55:...|              NULL|      true|
|      C0004|Customer_0004|         Active| 4 Main St|2025-12-31 00:55:...|              NULL|      true|
|      C0007|Customer_0007|         Active| 7 Main St|2026-02-04 23:12:...|              NULL|      true|
|      C0008|Customer_0008|       Inactive| 8 Main St|2025-12-31 00:55:...|              NULL|      true|
|      C0009|Customer_0009|         Active| 9 

##### **WHERE is_current = FALSE** #####

In [39]:
spark.sql("""
SELECT *
FROM dim_customer
WHERE is_current = FALSE
ORDER BY customer_id, effective_start_date
""").show()


StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 41, Finished, Available, Finished, False)

+-----------+-------------+---------------+----------+--------------------+--------------------+----------+
|customer_id|customer_name|customer_status|   address|effective_start_date|  effective_end_date|is_current|
+-----------+-------------+---------------+----------+--------------------+--------------------+----------+
|      C0005|Customer_0005|         Active| 5 Main St|2025-12-31 00:55:...|2026-02-04 23:16:...|     false|
|      C0006|Customer_0006|         Active| 6 Main St|2025-12-31 00:55:...|2026-02-04 23:16:...|     false|
|      C0007|Customer_0007|         Active| 7 Main St|2025-12-31 00:55:...|2025-12-31 01:39:...|     false|
|      C0011|Customer_0011|         Active|11 Main St|2025-12-31 00:55:...|2025-12-31 01:39:...|     false|
|      C0019|Customer_0019|         Active|19 Main St|2025-12-31 00:55:...|2025-12-31 01:39:...|     false|
|      C0021|Customer_0021|         Active|21 Main St|2025-12-31 00:55:...|2026-02-04 23:16:...|     false|
|      C0026|Customer_0026| 

In [40]:
# Count number of changes per customer

spark.sql("""
SELECT customer_id, COUNT(*) AS num_versions
FROM dim_customer
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY num_versions DESC
""").show()


StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 42, Finished, Available, Finished, False)

+-----------+------------+
|customer_id|num_versions|
+-----------+------------+
|      C0076|           2|
|      C0074|           2|
|      C0067|           2|
|      C0057|           2|
|      C0147|           2|
|      C0084|           2|
|      C0011|           2|
|      C0019|           2|
|      C0153|           2|
|      C0136|           2|
|      C0176|           2|
|      C0187|           2|
|      C0104|           2|
|      C0048|           2|
|      C0189|           2|
|      C0042|           2|
|      C0040|           2|
|      C0063|           2|
|      C0007|           2|
|      C0192|           2|
+-----------+------------+



#### <span style="background-color:pink;">Compare consecutive rows for each customer_id to see which attributes changed.

In [51]:
spark.sql("""
SELECT a.customer_id, a.customer_name, b.customer_name as customer_name_stg, a.customer_status, a.effective_start_date, a.effective_end_date
FROM dim_customer a left outer join (select * from src) b on b.customer_id = a.customer_id
WHERE a.is_current = FALSE
ORDER BY a.customer_id, a.effective_start_date
""").show()

# Compare consecutive rows for each customer_id to see which attributes changed.

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 53, Finished, Available, Finished, False)

+-----------+-------------+-------------------+---------------+--------------------+--------------------+
|customer_id|customer_name|  customer_name_stg|customer_status|effective_start_date|  effective_end_date|
+-----------+-------------+-------------------+---------------+--------------------+--------------------+
|      C0005|Customer_0005|Customer_0005_v2_v2|         Active|2025-12-31 00:55:...|2026-02-04 23:16:...|
|      C0006|Customer_0006|Customer_0006_v2_v2|         Active|2025-12-31 00:55:...|2026-02-04 23:16:...|
|      C0007|Customer_0007|      Customer_0007|         Active|2025-12-31 00:55:...|2025-12-31 01:39:...|
|      C0011|Customer_0011|      Customer_0011|         Active|2025-12-31 00:55:...|2025-12-31 01:39:...|
|      C0019|Customer_0019|      Customer_0019|         Active|2025-12-31 00:55:...|2025-12-31 01:39:...|
|      C0021|Customer_0021|Customer_0021_v2_v2|         Active|2025-12-31 00:55:...|2026-02-04 23:16:...|
|      C0026|Customer_0026|Customer_0026_v2_v2

In [49]:
spark.sql("select * from src").orderBy("customer_id").show()

StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 51, Finished, Available, Finished, False)

+-----------+-------------------+---------------+----------+--------------------+----------+
|customer_id|      customer_name|customer_status|   address|     effective_start|is_current|
+-----------+-------------------+---------------+----------+--------------------+----------+
|      C0001|      Customer_0001|         Active| 1 Main St|2026-02-04 23:14:...|      true|
|      C0002|      Customer_0002|       Inactive| 2 Main St|2026-02-04 23:14:...|      true|
|      C0003|      Customer_0003|         Active| 3 Main St|2026-02-04 23:14:...|      true|
|      C0004|      Customer_0004|         Active| 4 Main St|2026-02-04 23:14:...|      true|
|      C0005|Customer_0005_v2_v2|       Inactive|520 New St|2026-02-04 23:14:...|      true|
|      C0006|Customer_0006_v2_v2|       Inactive|976 New St|2026-02-04 23:14:...|      true|
|      C0007|      Customer_0007|         Active| 7 Main St|2026-02-04 23:14:...|      true|
|      C0008|      Customer_0008|       Inactive| 8 Main St|2026-02-04

In [43]:
spark.sql("""
SELECT customer_id, customer_name, customer_status, effective_start_date
FROM dim_customer
WHERE effective_end_date IS NOT NULL
ORDER BY effective_end_date DESC
""").show()
# Lists only the rows that were replaced by a new version.

StatementMeta(, 5c7737ce-3e3b-4c2e-9bb7-123e768d93c5, 45, Finished, Available, Finished, False)

+-----------+-------------+---------------+--------------------+
|customer_id|customer_name|customer_status|effective_start_date|
+-----------+-------------+---------------+--------------------+
+-----------+-------------+---------------+--------------------+



**Python / pandas approach (optional)**

**df_changes** now contains all actual changes in the dimension.


In [59]:
import pandas as pd

# Assume df is already loaded from Spark to Pandas
df = spark.table("dim_customer").toPandas()

# Sort by customer and effective date
df.sort_values(["customer_id", "effective_start_date"], inplace=True)

# Identify previous values per customer
df['prev_name'] = df.groupby('customer_id')['customer_name'].shift(1)
df['prev_status'] = df.groupby('customer_id')['customer_status'].shift(1)

# Filter only rows where something changed
df_changes = df[
    (df['customer_name'] != df['prev_name']) |
    (df['customer_status'] != df['prev_status'])
]

# Optional: view the first 40 rows
df_changes.head(40)



StatementMeta(, 4276fe30-aa96-4f52-a3ba-1fdc7f8a9cd8, 61, Finished, Available, Finished, False)

,customer_id,customer_name,customer_status,address,effective_start_date,effective_end_date,is_current,prev_name,prev_status
162,C0001,Customer_0001,Active,1 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
163,C0002,Customer_0002,Inactive,2 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
164,C0003,Customer_0003,Active,3 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
165,C0004,Customer_0004,Active,4 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
45,C0005,Customer_0005,Active,5 Main St,2025-12-31 00:55:52.504090,2026-02-04 23:16:19.297522,False,NaN,NaN
46,C0006,Customer_0006,Active,6 Main St,2025-12-31 00:55:52.504090,2026-02-04 23:16:19.297522,False,NaN,NaN
192,C0007,Customer_0007,Active,7 Main St,2025-12-31 00:55:52.504090,2025-12-31 01:39:15.219473,False,NaN,NaN
166,C0008,Customer_0008,Inactive,8 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
167,C0009,Customer_0009,Active,9 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN
168,C0010,Customer_0010,Active,10 Main St,2025-12-31 00:55:52.504090,NaT,True,NaN,NaN


### 7. **Return Success to Pipeline** ###

In [ ]:
from notebookutils import mssparkutils
mssparkutils.notebook.exit("SCD2_SUCCESS")

#### **This is a fully working SCD Type 2 Delta Lake notebook.** ###

It follows production best practices:

* Proper MERGE logic
* Versioning metadata
* Automatic start/end timestamps
* Change detection only on relevant columns
* Handles inserts + updates + new versions safely

#### Clear out dimension table for test

In [25]:
spark.sql(""" truncate table dim_customer """)

StatementMeta(, 5c7737ce-3e3b-4c2e-9bb7-123e768d93c5, 27, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]